# Notebook 17: Multi-Agent Systems -- Orchestration Patterns

**Sprint 3: Agentic AI** | Frontier ML Interview Prep Toolkit

---

Multi-agent systems are one of the most important emerging patterns in production AI. Consider, for example, a multi-agent RCA (root-cause-analysis) system). Now let's formalize the theory, learn the full taxonomy of patterns, and build a multi-agent system from scratch so you can speak fluently about it in interviews.

**Prerequisites**: Notebooks 13-16 (ReAct, tool use, memory, planning)

**Key connection**: A multi-agent RCA system (Task Planning Agent -> Top Level Agent -> RCA Agent) is a real production multi-agent system. This notebook gives you the vocabulary to discuss it at any frontier lab interview.

---
## 1. Self-Quiz (Active Recall)

**Before reading anything**, try to answer these from memory. Write your answers in the cell below, then check against the notebook content.

1. **Name 4 multi-agent architecture patterns.** What distinguishes each?
2. **When is multi-agent better than single agent?** Give specific criteria.
3. **How do agents communicate?** What are the options and tradeoffs?
4. **What is MCP (Model Context Protocol)?** Why does it matter for multi-agent systems?
5. **In a multi-agent RCA system, why use 3 agents instead of 1?** Frame this as a design decision with clear justification.

In [ ]:
# YOUR ANSWERS (write before reading further)
self_quiz_answers = {
    "four_patterns": "",
    "when_multi_agent": "",
    "communication_methods": "",
    "what_is_mcp": "",
    "camel_lens_why_3_agents": "",
}

# After completing the notebook, come back and grade yourself:
# How many did you get right? ___/5

---
## 2. Setup

In [ ]:
!pip install -q openai

In [ ]:
import json
import time
import uuid
import os
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable
from enum import Enum
from concurrent.futures import ThreadPoolExecutor, as_completed
import textwrap

# Optional: set your OpenAI API key for live demos
# os.environ["OPENAI_API_KEY"] = "sk-..."

# We'll build everything from scratch so you understand the internals.
# No multi-agent framework needed -- that's the point.

---
## 3. Why Multi-Agent?

### Single Agent Limitations

| Limitation | Description | Example |
|---|---|---|
| **Context window saturation** | One agent trying to hold all context (instructions, tools, history) hits limits | Analyzing very large of logs in one prompt |
| **Lack of specialization** | One system prompt can't make the model expert at everything | Being both a great planner AND a great code reviewer |
| **Reliability cliff** | As task complexity grows, single-agent accuracy degrades non-linearly | 10-step tool chains have ~35% success if each step is 90% |
| **No parallelism** | Sequential execution only; can't analyze multiple things at once | Processing 100 log files one at a time |
| **Debugging opacity** | When something fails, hard to know which part of the monolithic process broke | Was it the planning? The tool call? The synthesis? |

### Multi-Agent Benefits

| Benefit | Description | Example |
|---|---|---|
| **Divide and conquer** | Break complex tasks into manageable subtasks | Multi-agent RCA: planning, orchestration, and deep analysis as separate concerns |
| **Specialization** | Each agent has a focused system prompt and tool set | A "Researcher" agent vs a "Code Reviewer" agent |
| **Redundancy** | Multiple agents can cross-check each other's work | Two agents independently verify an answer |
| **Parallel execution** | Independent subtasks run simultaneously | Map-Reduce: analyze 20 log chunks in parallel |
| **Composability** | Agents can be reused across different systems | Same RCA agent used in a root-cause-analysis system and a monitoring system |

### When NOT to Use Multi-Agent

Multi-agent adds complexity. Don't use it when:
- **The task is simple**: If a single agent with a good prompt can do it, don't over-engineer.
- **Latency is critical**: Agent-to-agent communication adds overhead.
- **You can't observe/debug**: Multi-agent without good logging is a nightmare.
- **The task doesn't decompose**: Some tasks are inherently sequential and monolithic.

> **Interview framing**: "The key question with multi-agent systems is always: does the decomposition reduce overall error rate enough to justify the coordination overhead? Understanding both the power and the pitfalls matters more than adding agents for their own sake."

**Insider Tip:** Multi-agent is often unnecessary complexity. Anthropic's "Building Effective Agents" explicitly says: "start with the simplest architecture that works". In interviews, show you know WHEN to use multi-agent (genuinely different capabilities needed, parallelizable subtasks) vs WHEN to use a single agent with good tools.

---
## 4. Architecture Patterns

There are 5 canonical multi-agent patterns. Understanding when to use each is critical for system design interviews.

### Pattern 1: Supervisor / Worker

```
         +-----------+
         | Supervisor|
         +-----+-----+
          /    |    \\
    +-----+ +-----+ +-----+
    |Wkr 1| |Wkr 2| |Wkr 3|
    +-----+ +-----+ +-----+
```

**How it works**: One supervisor agent receives the task, decomposes it into subtasks, dispatches to specialized worker agents, collects results, and synthesizes a final answer.

**When to use**: 
- Tasks that naturally decompose into independent subtasks
- When you need centralized quality control
- When workers have distinct specializations

**Example**: A multi-agent RCA system -- Task Planning Agent (supervisor) decomposes, RCA Agents (workers) analyze specific log segments.

**Tradeoffs**: Supervisor is a bottleneck; if it makes a bad plan, all workers suffer. But centralized control makes debugging easier.

---

### Pattern 2: Debate / Consensus

```
    +-----+     +-----+     +-----+
    |Agt A| <-> |Agt B| <-> |Agt C|
    +-----+     +-----+     +-----+
         \\       |        /
          +------+------+
          |  Consensus  |
          +-------------+
```

**How it works**: Multiple agents independently analyze the same problem, then debate their answers. A judge (or voting mechanism) selects the best answer.

**When to use**:
- High-stakes decisions where errors are costly
- Tasks where there's genuine ambiguity
- When you want to reduce hallucination risk

**Example**: Code review -- 3 agents review the same PR, debate issues, converge on a final review.

**Tradeoffs**: High cost (N agents x full context), but much more reliable for critical decisions.

---

### Pattern 3: Pipeline (Sequential)

```
    +-----+    +-----+    +-----+    +-----+
    |Agt 1| -> |Agt 2| -> |Agt 3| -> |Agt 4|
    +-----+    +-----+    +-----+    +-----+
    Research    Analyze    Draft      Review
```

**How it works**: Agents are arranged in sequence. Each agent takes the previous agent's output, transforms it, and passes it to the next. Like Unix pipes.

**When to use**:
- Tasks with clear sequential stages
- When each stage requires different expertise
- Data processing workflows

**Example**: Content creation -- Researcher gathers info -> Analyst identifies key themes -> Writer drafts article -> Editor polishes.

**Tradeoffs**: Simple to understand and debug, but no parallelism. Error in early stages propagates.

---

### Pattern 4: Specialization (Team)

```
    +----------+  +--------+  +----------+
    |Researcher|  | Coder  |  | Reviewer |
    +----+-----+  +---+----+  +----+-----+
         |            |            |
         +------+-----+------+----+
                | Shared Mem  |
                +-------------+
```

**How it works**: Each agent has a deep specialization (domain, tools, personality). They collaborate via shared memory or message passing. No strict hierarchy.

**When to use**:
- Complex projects requiring diverse expertise
- When the interaction pattern isn't predictable in advance
- Software development (researcher, coder, tester, reviewer)

**Example**: AI software team -- PM defines requirements, Developer writes code, QA tests, DevOps deploys.

**Tradeoffs**: Most flexible but hardest to coordinate. Needs good communication protocols.

---

### Pattern 5: Map-Reduce

```
           +--------+
           | Mapper |
           +---+----+
          /    |    \\
    +---+  +---+  +---+
    |W 1|  |W 2|  |W 3|   <- parallel
    +---+  +---+  +---+
          \\   |   /
          +---+---+
          |Reducer|
          +-------+
```

**How it works**: A mapper splits the input into chunks, worker agents process chunks in parallel, a reducer merges the results.

**When to use**:
- Large inputs that exceed a single context window
- Embarrassingly parallel analysis tasks
- When the merge operation is well-defined

**Example**: Multi-agent RCA log analysis -- Split very large logs into chunks, analyze each chunk for anomalies in parallel, merge findings into a unified RCA report.

**Tradeoffs**: Great throughput, but the reduce step can lose information. Good for summarization, risky for tasks requiring global context.

---
## 5. Multi-Agent RCA as Case Study

A production multi-agent system, re-framed in the vocabulary of multi-agent research.

### Architecture

```
    User Query ("Why did service X fail?")
            |
            v
    +-------------------+
    | Task Planning Agent|  <-- Supervisor pattern
    | (Decomposes into   |      Breaks complex RCA into analysis steps
    |  analysis steps)   |      Decides which logs/metrics to examine
    +---------+---------+
              |
              v
    +-------------------+
    | Top Level Agent    |  <-- Orchestrator / Pipeline pattern
    | (Orchestrates      |      Routes to RCA agents, manages flow
    |  execution)        |      Handles the map-reduce for large logs
    +---------+---------+
              |
        +-----+-----+
        |     |     |
        v     v     v
    +-----+ +-----+ +-----+
    |RCA 1| |RCA 2| |RCA 3|  <-- Map-Reduce pattern
    |Chunk| |Chunk| |Chunk|      Parallel analysis of log segments
    +-----+ +-----+ +-----+
        \\     |     /
         v    v    v
    +-------------------+
    | Merge & Synthesize|  <-- Reduce step
    | (Unified RCA)     |
    +-------------------+
```

### Key Design Decisions (Interview Framing)

| Decision | Why |
|---|---|
| **3 agents, not 1** | Separation of concerns: planning, orchestration, and deep analysis are fundamentally different skills. One agent trying to do all three hits context window limits and loses accuracy. |
| **Task Planning separate from execution** | Planning requires high-level reasoning about what to investigate. Execution requires deep domain knowledge about log formats. Mixing them degrades both. |
| **Map-Reduce for large logs** | very large logs can't fit in a single context window. Chunking + parallel analysis gives both coverage and speed. The reduce step merges findings while resolving contradictions. |
| **Self-hosted model (data-privacy requirement)** | Production requirement: data stays within the organization's boundary. The self-hosted infrastructure provides enterprise SLAs. The internal LLM was chosen for its strong instruction following and tool use. |

### Results (Quantified Impact)

- **a major time reduction**: RCA that took engineers hours now takes minutes
- **a strong ROI**: Cost of LLM inference vs. senior engineer time saved
- **2-10 min vs hours**: Consistent, reproducible analysis at any time of day

### How to Talk About This in an Interview

> A representative design: a multi-agent root cause analysis system using a **supervisor/worker architecture** with three specialized agents -- a Task Planning agent that decomposes complex investigations into analysis steps, an orchestration agent that manages the map-reduce flow over large log files, and analysis agents that examine specific log segments in parallel.
>
> The key design insight was **separation of concerns** -- planning, orchestration, and analysis require fundamentally different prompt strategies and tool sets. By splitting them, each agent could be optimized independently, and we could scale horizontally for large logs using map-reduce.
>
> Such a system can process very large log files by running analysis agents in parallel, cutting diagnosis time from hours to minutes."

**Practice this out loud. Time yourself. It should take 45-60 seconds.**

**Insider Tip:** A strong STAR project for agentic AI interviews is a multi-agent system with clear, quantified impact -- a meaningful time reduction and a strong ROI. If you have built something like this, use it as your STAR project for agentic AI interviews. Practice the narrative: problem (manual RCA taking hours) --> architecture (3-agent supervisor/worker with task planning, log analysis, root cause synthesis) --> why multi-agent was the right choice (genuinely different capabilities: planning vs. log parsing vs. causal reasoning) --> results (80% faster, a strong ROI).

---
## 6. Build a Multi-Agent System from Scratch

We'll build a complete multi-agent framework, then use it to create a 3-agent research team.

In [ ]:
# ============================================================
# Base Agent Class
# ============================================================

class Agent:
    """Base agent with a name, role, system prompt, tools, and generation."""
    
    def __init__(self, name: str, role: str, system_prompt: str, 
                 tools: Optional[List[Dict]] = None, 
                 model: str = "gpt-4o-mini"):
        self.name = name
        self.role = role
        self.system_prompt = system_prompt
        self.tools = tools or []
        self.model = model
        self.conversation_history: List[Dict] = []
        self.token_usage = {"prompt_tokens": 0, "completion_tokens": 0}
    
    def generate(self, user_message: str, temperature: float = 0.7) -> str:
        """Generate a response. Uses OpenAI if available, otherwise simulates."""
        self.conversation_history.append({"role": "user", "content": user_message})
        
        try:
            from openai import OpenAI
            client = OpenAI()
            messages = [{"role": "system", "content": self.system_prompt}] + self.conversation_history
            
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
            )
            
            reply = response.choices[0].message.content
            self.token_usage["prompt_tokens"] += response.usage.prompt_tokens
            self.token_usage["completion_tokens"] += response.usage.completion_tokens
        except Exception:
            # Simulate response for demonstration
            reply = self._simulate_response(user_message)
        
        self.conversation_history.append({"role": "assistant", "content": reply})
        return reply
    
    def _simulate_response(self, user_message: str) -> str:
        """Simulate a response when no API key is available."""
        return f"[{self.name} ({self.role})]: Processed task: {user_message[:100]}..."
    
    def reset(self):
        """Clear conversation history for a fresh start."""
        self.conversation_history = []
    
    def get_cost_estimate(self, price_per_1k_prompt=0.00015, price_per_1k_completion=0.0006):
        """Estimate cost based on token usage (gpt-4o-mini pricing)."""
        prompt_cost = (self.token_usage["prompt_tokens"] / 1000) * price_per_1k_prompt
        completion_cost = (self.token_usage["completion_tokens"] / 1000) * price_per_1k_completion
        return {"prompt_cost": prompt_cost, "completion_cost": completion_cost, 
                "total_cost": prompt_cost + completion_cost}
    
    def __repr__(self):
        return f"Agent(name='{self.name}', role='{self.role}')"


# Quick test
test_agent = Agent(
    name="TestAgent",
    role="tester",
    system_prompt="You are a test agent. Respond briefly."
)
print(test_agent)
print(test_agent.generate("Hello, are you working?"))

In [ ]:
# ============================================================
# Supervisor Agent
# ============================================================

class Supervisor(Agent):
    """A supervisor agent that decomposes tasks, dispatches to workers, 
    and synthesizes results."""
    
    def __init__(self, name: str, system_prompt: str, model: str = "gpt-4o-mini"):
        super().__init__(name=name, role="supervisor", system_prompt=system_prompt, model=model)
        self.workers: Dict[str, 'Worker'] = {}
    
    def register_worker(self, worker: 'Worker'):
        """Register a worker agent."""
        self.workers[worker.name] = worker
        print(f"  Registered worker: {worker.name} ({worker.role})")
    
    def decompose_task(self, task: str) -> List[Dict[str, str]]:
        """Break a task into subtasks assigned to specific workers."""
        worker_list = ", ".join([f"{w.name} ({w.role})" for w in self.workers.values()])
        
        decomposition_prompt = f"""You are a task planning supervisor. You have these workers available:
{worker_list}

Break the following task into 2-4 subtasks. For each subtask, specify which worker should handle it.

Respond in this exact JSON format:
[{{"worker": "WorkerName", "subtask": "description of what they should do"}}]

Task: {task}"""
        
        response = self.generate(decomposition_prompt)
        
        try:
            # Try to parse JSON from the response
            import re
            json_match = re.search(r'\[.*\]', response, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
        except (json.JSONDecodeError, AttributeError):
            pass
        
        # Fallback: create a default decomposition
        subtasks = []
        for worker_name, worker in self.workers.items():
            subtasks.append({
                "worker": worker_name,
                "subtask": f"As a {worker.role}, contribute to: {task}"
            })
        return subtasks
    
    def synthesize(self, task: str, results: Dict[str, str]) -> str:
        """Combine worker results into a final answer."""
        results_text = "\n\n".join([
            f"=== {worker} ===\n{result}" 
            for worker, result in results.items()
        ])
        
        synthesis_prompt = f"""You are synthesizing results from your team to answer the original task.

Original task: {task}

Worker results:
{results_text}

Provide a comprehensive, well-organized final answer that integrates all worker contributions.
Resolve any contradictions. Highlight key insights."""
        
        return self.generate(synthesis_prompt)


# ============================================================
# Worker Agent
# ============================================================

class Worker(Agent):
    """A worker agent that receives subtasks and executes them."""
    
    def __init__(self, name: str, role: str, system_prompt: str,
                 tools: Optional[List[Dict]] = None, model: str = "gpt-4o-mini"):
        super().__init__(name=name, role=role, system_prompt=system_prompt, 
                        tools=tools, model=model)
    
    def execute(self, subtask: str) -> str:
        """Execute a subtask and return the result."""
        self.reset()  # Fresh context for each subtask
        return self.generate(subtask)


print("Supervisor and Worker classes defined.")

In [ ]:
# ============================================================
# Multi-Agent System Orchestrator
# ============================================================

import copy

class MultiAgentSystem:
    """Orchestrates a multi-agent system with a supervisor and workers."""
    
    def __init__(self, name: str, supervisor: Supervisor, 
                 parallel: bool = False, verbose: bool = True):
        self.name = name
        self.supervisor = supervisor
        self.parallel = parallel
        self.verbose = verbose
        self.execution_log: List[Dict] = []
    
    def register_agent(self, agent: Worker):
        """Add a worker agent to the system."""
        self.supervisor.register_worker(agent)
    
    def run(self, task: str) -> str:
        """Full execution: decompose -> dispatch -> collect -> synthesize."""
        start_time = time.time()
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"Multi-Agent System: {self.name}")
            print(f"Task: {task}")
            print(f"{'='*60}")
        
        # Step 1: Decompose
        if self.verbose:
            print(f"\n[1] Supervisor decomposing task...")
        subtasks = self.supervisor.decompose_task(task)
        
        if self.verbose:
            print(f"    Generated {len(subtasks)} subtasks:")
            for i, st in enumerate(subtasks):
                print(f"    {i+1}. [{st['worker']}] {st['subtask'][:80]}...")
        
        # Step 2: Dispatch and execute
        if self.verbose:
            print(f"\n[2] Dispatching to workers {'(parallel)' if self.parallel else '(sequential)'}...")
        
        # Use a list of (worker_name, subtask_desc, result) to avoid
        # overwriting results when a worker handles multiple subtasks.
        results = {}
        
        if self.parallel:
            # Parallel execution
            # BUG FIX: Worker.execute() mutates self.conversation_history via
            # reset() and generate(). If the same worker is assigned multiple
            # subtasks, concurrent threads would corrupt shared state.
            # Solution: deep-copy workers so each thread gets its own instance.
            with ThreadPoolExecutor(max_workers=len(subtasks)) as executor:
                futures = {}
                for idx, st in enumerate(subtasks):
                    worker_name = st["worker"]
                    if worker_name in self.supervisor.workers:
                        # Deep-copy to avoid race condition on conversation_history
                        worker_copy = copy.deepcopy(self.supervisor.workers[worker_name])
                        usage_baseline = dict(worker_copy.token_usage)
                        future = executor.submit(worker_copy.execute, st["subtask"])
                        # Use (worker_name, idx) as key to handle duplicate workers
                        futures[future] = (worker_name, idx, worker_copy, usage_baseline)
                
                for future in as_completed(futures):
                    worker_name, idx, worker_copy, usage_baseline = futures[future]
                    result_key = worker_name if worker_name not in results else f"{worker_name}_subtask{idx}"
                    try:
                        results[result_key] = future.result()
                    except Exception as e:
                        results[result_key] = f"ERROR: {str(e)}"
                    # BUG FIX: token_usage accumulated on the deep copies was discarded,
                    # so get_total_cost() undercounted. Merge the copies' new usage back
                    # into the original workers (delta over the baseline at copy time).
                    original = self.supervisor.workers[worker_name]
                    for k in ("prompt_tokens", "completion_tokens"):
                        original.token_usage[k] += worker_copy.token_usage[k] - usage_baseline[k]
        else:
            # Sequential execution
            for idx, st in enumerate(subtasks):
                worker_name = st["worker"]
                if worker_name in self.supervisor.workers:
                    worker = self.supervisor.workers[worker_name]
                    if self.verbose:
                        print(f"    Executing: {worker_name} -> {st['subtask'][:60]}...")
                    result_key = worker_name if worker_name not in results else f"{worker_name}_subtask{idx}"
                    results[result_key] = worker.execute(st["subtask"])
                else:
                    if self.verbose:
                        print(f"    WARNING: Worker '{worker_name}' not found, skipping.")
        
        # Step 3: Synthesize
        if self.verbose:
            print(f"\n[3] Supervisor synthesizing results...")
        final_answer = self.supervisor.synthesize(task, results)
        
        elapsed = time.time() - start_time
        
        # Log execution
        self.execution_log.append({
            "task": task,
            "subtasks": subtasks,
            "results": results,
            "final_answer": final_answer,
            "elapsed_seconds": elapsed,
        })
        
        if self.verbose:
            print(f"\n[Done] Completed in {elapsed:.2f}s")
            print(f"{'='*60}")
        
        return final_answer
    
    def get_total_cost(self):
        """Get total cost across all agents."""
        total = 0.0
        costs = {}
        
        sup_cost = self.supervisor.get_cost_estimate()
        costs[self.supervisor.name] = sup_cost
        total += sup_cost["total_cost"]
        
        for name, worker in self.supervisor.workers.items():
            w_cost = worker.get_cost_estimate()
            costs[name] = w_cost
            total += w_cost["total_cost"]
        
        return {"per_agent": costs, "total": total}


print("MultiAgentSystem class defined.")

In [ ]:
# ============================================================
# Build a 3-Agent Research Team
# ============================================================

# Create the supervisor
supervisor = Supervisor(
    name="ProjectLead",
    system_prompt="""You are a project lead who coordinates a research team.
You decompose research tasks into subtasks for your team members.
You are skilled at identifying what each team member should focus on
and synthesizing their findings into a coherent final report."""
)

# Create specialized workers
researcher = Worker(
    name="Researcher",
    role="researcher",
    system_prompt="""You are a thorough researcher. Given a research question,
you identify key papers, findings, and arguments. You cite sources
and present balanced perspectives. Focus on factual accuracy."""
)

analyst = Worker(
    name="Analyst",
    role="analyst",
    system_prompt="""You are a critical analyst. Given information or a topic,
you identify patterns, compare approaches, evaluate tradeoffs,
and provide structured analysis. Use tables and comparisons when helpful."""
)

writer = Worker(
    name="Writer",
    role="writer",
    system_prompt="""You are a skilled technical writer. Given research findings
and analysis, you create clear, well-structured summaries that
are accessible to a technical audience. Focus on clarity and insight."""
)

# Assemble the system
research_team = MultiAgentSystem(
    name="Research Team",
    supervisor=supervisor,
    parallel=False,  # Sequential for clearer demonstration
    verbose=True
)
research_team.register_agent(researcher)
research_team.register_agent(analyst)
research_team.register_agent(writer)

print("\nResearch team assembled!")

In [ ]:
# ============================================================
# Run the Multi-Agent System
# ============================================================

task = """Research and summarize the current state of the RLHF vs DPO debate.
Cover: key differences, when to use each, recent results, and where the field is heading."""

result = research_team.run(task)

print("\n" + "="*60)
print("FINAL OUTPUT")
print("="*60)
print(result)

# Show cost breakdown
costs = research_team.get_total_cost()
print(f"\nCost breakdown:")
for agent_name, cost in costs["per_agent"].items():
    print(f"  {agent_name}: ${cost['total_cost']:.6f}")
print(f"  TOTAL: ${costs['total']:.6f}")

---
## 7. Communication Protocols

How agents talk to each other is a fundamental design decision. Let's implement both major approaches.

In [ ]:
# ============================================================
# Message-Based Communication
# ============================================================

class MessageType(Enum):
    TASK = "task"              # Assign a task to an agent
    RESULT = "result"          # Return a result from a task
    QUESTION = "question"      # Ask another agent for clarification
    CLARIFICATION = "clarification"  # Respond to a question
    HANDOFF = "handoff"        # Transfer context to another agent


@dataclass
class Message:
    """A message between agents."""
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    sender: str = ""
    receiver: str = ""
    content: str = ""
    message_type: MessageType = MessageType.TASK
    timestamp: float = field(default_factory=time.time)
    metadata: Dict = field(default_factory=dict)
    
    def __repr__(self):
        return (f"Message({self.message_type.value}: "
                f"{self.sender} -> {self.receiver}, "
                f"'{self.content[:50]}...')")


class MessageBus:
    """Routes messages between agents and logs all communication."""
    
    def __init__(self):
        self.agents: Dict[str, Agent] = {}
        self.message_log: List[Message] = []
        self.inboxes: Dict[str, List[Message]] = {}  # agent_name -> messages
    
    def register(self, agent: Agent):
        """Register an agent with the message bus."""
        self.agents[agent.name] = agent
        self.inboxes[agent.name] = []
    
    def send(self, message: Message):
        """Send a message from one agent to another."""
        self.message_log.append(message)
        
        if message.receiver in self.inboxes:
            self.inboxes[message.receiver].append(message)
        else:
            print(f"  WARNING: Receiver '{message.receiver}' not found")
    
    def receive(self, agent_name: str) -> List[Message]:
        """Get all pending messages for an agent."""
        messages = self.inboxes.get(agent_name, [])
        self.inboxes[agent_name] = []  # Clear inbox
        return messages
    
    def get_conversation_trace(self) -> str:
        """Get a human-readable trace of all messages."""
        lines = []
        for msg in self.message_log:
            lines.append(
                f"[{msg.message_type.value.upper():15s}] "
                f"{msg.sender:12s} -> {msg.receiver:12s}: "
                f"{msg.content[:80]}"
            )
        return "\n".join(lines)


# Demonstrate message-based communication
bus = MessageBus()

# Register agents
agents = {
    "planner": Agent("Planner", "planner", "You plan tasks."),
    "executor": Agent("Executor", "executor", "You execute tasks."),
    "reviewer": Agent("Reviewer", "reviewer", "You review results."),
}
for a in agents.values():
    bus.register(a)

# Simulate a workflow
bus.send(Message(
    sender="Planner", receiver="Executor",
    content="Analyze the error logs from service-auth for the last 24 hours",
    message_type=MessageType.TASK
))

bus.send(Message(
    sender="Executor", receiver="Planner",
    content="Found 3 recurring NullPointerException in AuthService.validate()",
    message_type=MessageType.RESULT
))

bus.send(Message(
    sender="Planner", receiver="Reviewer",
    content="Review this finding: NullPointerException in AuthService.validate()",
    message_type=MessageType.TASK
))

bus.send(Message(
    sender="Reviewer", receiver="Executor",
    content="Can you check if this started after the last deploy?",
    message_type=MessageType.QUESTION
))

bus.send(Message(
    sender="Executor", receiver="Reviewer",
    content="Yes, errors began exactly at deploy timestamp 2024-01-15T14:30:00Z",
    message_type=MessageType.CLARIFICATION
))

print("Message Trace:")
print(bus.get_conversation_trace())

In [ ]:
# ============================================================
# Shared Memory Communication
# ============================================================

class SharedMemory:
    """A shared workspace where agents can read and write."""
    
    def __init__(self):
        self.store: Dict[str, Any] = {}
        self.access_log: List[Dict] = []
    
    def write(self, agent_name: str, key: str, value: Any):
        """Write a value to shared memory."""
        self.store[key] = value
        self.access_log.append({
            "agent": agent_name, "action": "write", 
            "key": key, "timestamp": time.time()
        })
    
    def read(self, agent_name: str, key: str) -> Any:
        """Read a value from shared memory."""
        self.access_log.append({
            "agent": agent_name, "action": "read",
            "key": key, "timestamp": time.time()
        })
        return self.store.get(key, None)
    
    def read_all(self, agent_name: str) -> Dict[str, Any]:
        """Read all shared memory."""
        self.access_log.append({
            "agent": agent_name, "action": "read_all",
            "timestamp": time.time()
        })
        return dict(self.store)


# Demonstrate shared memory
memory = SharedMemory()

# Researcher writes findings
memory.write("Researcher", "findings", {
    "topic": "RLHF vs DPO",
    "key_papers": ["InstructGPT", "DPO (Rafailov 2023)", "KTO"],
    "main_finding": "DPO eliminates reward model but may be less robust"
})

# Analyst reads findings and adds analysis
findings = memory.read("Analyst", "findings")
memory.write("Analyst", "analysis", {
    "comparison": {
        "RLHF": {"pros": ["more flexible", "proven at scale"], "cons": ["complex", "unstable"]},
        "DPO": {"pros": ["simple", "stable"], "cons": ["less flexible", "needs good data"]}
    },
    "recommendation": "Use DPO for most cases, RLHF when reward modeling matters"
})

# Writer reads everything and produces report
all_data = memory.read_all("Writer")
print("Shared Memory Contents:")
for key, value in all_data.items():
    print(f"\n  [{key}]:")
    print(f"    {json.dumps(value, indent=4)[:200]}")

print(f"\nAccess log ({len(memory.access_log)} entries):")
for entry in memory.access_log:
    print(f"  {entry['agent']:12s} {entry['action']:10s} {entry.get('key', 'ALL')}")

In [ ]:
# ============================================================
# Agent Handoff
# ============================================================

class AgentHandoff:
    """Manages transferring context from one agent to another."""
    
    @staticmethod
    def create_handoff_context(source_agent: Agent, 
                                summary: str,
                                key_findings: List[str],
                                next_steps: List[str]) -> Dict:
        """Package context for handoff."""
        return {
            "from_agent": source_agent.name,
            "from_role": source_agent.role,
            "summary": summary,
            "key_findings": key_findings,
            "next_steps": next_steps,
            "conversation_length": len(source_agent.conversation_history),
            "timestamp": time.time(),
        }
    
    @staticmethod
    def format_for_receiving_agent(handoff: Dict) -> str:
        """Format handoff context as a prompt for the receiving agent."""
        findings = "\n".join(f"  - {f}" for f in handoff["key_findings"])
        steps = "\n".join(f"  - {s}" for s in handoff["next_steps"])
        
        return f"""=== HANDOFF FROM {handoff['from_agent']} ({handoff['from_role']}) ===

Summary: {handoff['summary']}

Key Findings:
{findings}

Suggested Next Steps:
{steps}

Please continue the analysis from here."""


# Demonstrate handoff
research_agent = Agent("ResearchBot", "researcher", "You research topics.")
analysis_agent = Agent("AnalysisBot", "analyst", "You analyze findings.")

# Research agent does its work...
handoff = AgentHandoff.create_handoff_context(
    source_agent=research_agent,
    summary="Completed initial research on RLHF vs DPO alignment methods.",
    key_findings=[
        "DPO eliminates the need for a separate reward model",
        "RLHF is more flexible but requires PPO training loop",
        "Recent work (KTO, IPO, ORPO) builds on DPO's simplicity",
        "DeepSeek-R1 uses GRPO, a variant closer to RLHF"
    ],
    next_steps=[
        "Compare computational costs quantitatively",
        "Analyze which method works better with limited preference data",
        "Evaluate hybrid approaches"
    ]
)

# Format for the analysis agent
handoff_prompt = AgentHandoff.format_for_receiving_agent(handoff)
print(handoff_prompt)

### Comparison: Message Passing vs Shared Memory

| Aspect | Message Passing | Shared Memory |
|---|---|---|
| **Communication** | Explicit, point-to-point | Implicit, any agent reads/writes |
| **Traceability** | High -- every message logged | Medium -- need access logs |
| **Coupling** | Loose -- agents don't know internals | Tighter -- agents share data format |
| **Scalability** | Scales well, messages are lightweight | Can bottleneck on shared state |
| **Use when** | Clear sender/receiver relationships | Agents need to see each other's work |
| **Example** | Multi-agent RCA: Task Planning -> RCA Agent | Shared workspace for a coding team |

In practice, production systems often use **both**: message passing for task assignment and shared memory for accumulated state.

---
## 8. MCP: Model Context Protocol

### What is MCP?

MCP (Model Context Protocol) is an **open standard** for connecting AI agents to external tools and data sources, created by Anthropic (Nov 2024); governance moved to the Linux Foundation ecosystem in December 2025. Think of it as "USB-C for AI" -- a universal interface.

### Why Does MCP Exist?

Before MCP, every tool integration was custom:
- Agent A uses OpenAI function calling format
- Agent B uses Anthropic tool use format
- Agent C uses LangChain tools
- Tool provider has to implement N different integrations

MCP standardizes this into one protocol.

### Architecture

```
  +------------------+
  |   Your App       |  <-- HOST (e.g., Claude Desktop, an IDE, your agent system)
  |  +------------+  |
  |  | MCP Client |  |  <-- CLIENT (lives inside the host, one per server connection)
  |  +------+-----+  |
  +---------|--------+
            | (MCP Protocol: JSON-RPC over stdio or Streamable HTTP)
  +---------|--------+
  |  +------+-----+  |
  |  | MCP Server |  |  <-- SERVER (tool/data provider)
  |  +------------+  |
  |  Postgres, GitHub,|
  |  Slack, etc.      |
  +------------------+
```

*Transports*: stdio (local servers) and Streamable HTTP (remote servers); the original HTTP+SSE transport was deprecated in March 2025.

### Three Primitives

| Primitive | What it does | Direction | Example |
|---|---|---|---|
| **Tools** | Actions the agent can take | Client -> Server | `search_database(query)`, `create_issue(title, body)` |
| **Resources** | Data the agent can read | Client -> Server | `file://logs/error.log`, `db://users/table` |
| **Prompts** | Templates for common interactions | Server -> Client | "Summarize this PR" template with slots |

### Why MCP Matters for Multi-Agent Systems

1. **Interoperability**: Agents from different providers can use the same tools
2. **Composability**: Plug any MCP-compatible tool into any MCP-compatible agent
3. **Ecosystem**: Growing library of MCP servers (databases, APIs, file systems)
4. **Standards-based**: Open protocol means no vendor lock-in

### Connection to Your Work

In a multi-agent RCA system, each agent accesses different tools (monitoring services, log stores, dashboards). With MCP, these integrations could be standardized MCP servers that any agent in your system connects to, rather than custom code per tool per agent.

### Interview Talking Point

> "MCP is important because it solves the N x M problem -- N agents needing M tools used to require N*M custom integrations. With MCP, each tool implements the protocol once, and any agent can use it. This is critical for production multi-agent systems where you want to swap or upgrade individual agents without rewriting tool integrations."

---
## 9. Production Considerations

### Cost Management

Multi-agent systems multiply API costs. Each agent consumes tokens for:
- System prompt (repeated every call)
- Conversation history (grows over time)
- Tool calling overhead

**Strategies**:
- Track token usage per agent (we implemented this in our `Agent` class)
- Use cheaper models for simple workers, expensive models only for the supervisor
- Limit conversation history length per agent
- Cache common queries

### Latency: Parallel vs Sequential

| Pattern | Latency | When |
|---|---|---|
| Sequential | Sum of all agent latencies | Agents depend on each other's output |
| Parallel | Max of all agent latencies | Independent subtasks (map-reduce) |
| Hybrid | Depends on DAG structure | Most real systems |

For a multi-agent RCA system: Task Planning (sequential) -> Map-Reduce analysis (parallel) -> Synthesis (sequential). Total latency = planning + max(chunk analyses) + synthesis.

### Error Propagation

**One agent's failure should NOT crash the system.**

Strategies:
1. **Retry with backoff**: If an agent fails, retry 2-3 times with exponential backoff
2. **Fallback agents**: If the primary agent fails, route to a backup
3. **Graceful degradation**: If a worker fails, the supervisor synthesizes from available results
4. **Circuit breaker**: If an agent fails repeatedly, stop sending it tasks

### Observability

Multi-agent systems are hard to debug without good observability.

Must-haves:
- **Message tracing**: Log every agent-to-agent communication (we built the MessageBus for this)
- **Token counting**: Per-agent, per-request token usage
- **Latency tracking**: How long each agent takes
- **Decision logging**: Why the supervisor made each routing decision
- **Error categorization**: Which agents fail, on what types of tasks

### Scaling

Production multi-agent systems need:
- **Async execution**: Don't block on slow agents
- **Rate limiting**: Respect API limits per agent
- **Caching**: Don't re-analyze the same log chunk twice
- **State persistence**: Save intermediate results so you can resume

In [ ]:
# ============================================================
# Production-Ready Error Handling Demo
# ============================================================

import random

class ResilientWorker(Worker):
    """A worker with retry logic and fallback behavior."""
    
    def __init__(self, *args, failure_rate: float = 0.3, max_retries: int = 3, **kwargs):
        super().__init__(*args, **kwargs)
        self.failure_rate = failure_rate
        self.max_retries = max_retries
        self.failure_count = 0
        self.retry_count = 0
    
    def execute(self, subtask: str) -> str:
        """Execute with retry logic."""
        for attempt in range(self.max_retries):
            try:
                # Simulate random failures
                if random.random() < self.failure_rate:
                    raise RuntimeError(f"Agent {self.name} failed (simulated)")
                
                result = super().execute(subtask)
                return result
                
            except Exception as e:
                self.retry_count += 1
                if attempt < self.max_retries - 1:
                    wait_time = 2 ** attempt * 0.1  # Exponential backoff (fast for demo)
                    print(f"    RETRY: {self.name} failed (attempt {attempt+1}), "
                          f"retrying in {wait_time:.1f}s...")
                    time.sleep(wait_time)
                else:
                    self.failure_count += 1
                    return f"[FALLBACK] {self.name} failed after {self.max_retries} attempts. " \
                           f"Partial result: Unable to complete subtask."


# Demo with resilient workers
resilient_worker = ResilientWorker(
    name="ResilientAnalyst",
    role="analyst",
    system_prompt="You analyze data.",
    failure_rate=0.4,  # 40% chance of failure per attempt
)

print("Running 5 tasks with a 40% failure rate per attempt:")
for i in range(5):
    result = resilient_worker.execute(f"Analyze dataset {i}")
    status = "FALLBACK" if "[FALLBACK]" in result else "SUCCESS"
    print(f"  Task {i}: {status}")

print(f"\nStats: {resilient_worker.retry_count} retries, "
      f"{resilient_worker.failure_count} final failures")

---
## 10. "Why Does This Work?" -- Deep Questions

### Q: Supervisor vs peer architectures -- when should you use each?

**Supervisor (hierarchical)** when:
- Tasks have a natural decomposition
- You need centralized quality control
- Debugging matters (clear audit trail)
- Workers don't need to communicate with each other

**Peer (flat)** when:
- The interaction pattern is unpredictable
- Agents need to iterate/debate
- No single agent has the "right" decomposition
- The task benefits from diverse perspectives

**Key insight**: Most production systems are hierarchical because they're easier to debug. Research systems are often peer-based because they're exploring new patterns.

---

### Q: How do you handle agent disagreement?

Options (from simple to sophisticated):
1. **Majority vote**: Take the most common answer (cheap, simple)
2. **Confidence-weighted vote**: Agents report confidence, weight by it
3. **Judge agent**: A separate agent evaluates all proposals
4. **Iterative debate**: Agents see each other's answers and iterate until convergence
5. **Ensemble**: Combine all answers (e.g., union of findings rather than picking one)

**For a multi-agent RCA system**: The supervisor (Top Level Agent) resolves disagreements during the reduce step. If two RCA agents find different root causes in different log chunks, the supervisor evaluates which findings are causally connected and synthesizes a coherent narrative.

---

### Q: What's the overhead of multi-agent vs single agent with more tools?

| Factor | Multi-Agent | Single Agent + Tools |
|---|---|---|
| Token cost | Higher (N system prompts, N histories) | Lower (1 system prompt, 1 history) |
| Latency | Higher (coordination overhead) | Lower (no inter-agent communication) |
| Specialization | High (each agent is focused) | Lower (one prompt does everything) |
| Reliability | Higher (failures are isolated) | Lower (one failure cascades) |
| Debugging | Easier (clear agent boundaries) | Harder (monolithic trace) |

**Rule of thumb**: Start with a single agent. Switch to multi-agent when:
- The single agent's accuracy drops below your threshold
- You need parallel processing
- The task naturally decomposes into >3 distinct phases
- You need different models for different subtasks

---
## Interview Question Bank

*Multi-agent questions are where interviews get interesting -- and where candidates most often over-engineer. The senior-level insight is knowing when NOT to use multi-agent.*

---

### Q1: "Design a multi-agent system for automated code review" -- SYSTEM DESIGN (45 min)

**What this tests**: Full system design ability. Can you decompose a problem into agents, define their interactions, and handle edge cases?

**Good answer** (hire):
- Reviewer agent reads the diff and provides feedback
- Author agent (or the human) responds to feedback
- Clear interaction protocol (reviewer comments, author addresses, reviewer re-reviews)

**Great answer** (strong hire): Specialized agents with orchestration:

**Architecture**:
- **Orchestrator**: routes the PR to relevant reviewers based on file types and change patterns
- **Security Reviewer**: focuses exclusively on security issues (SQL injection, XSS, auth bypass, secrets in code)
- **Style Reviewer**: checks code style, naming conventions, documentation
- **Logic Reviewer**: analyzes correctness, edge cases, error handling
- **Test Coverage Reviewer**: checks if changes have adequate test coverage
- **Synthesizer**: aggregates all reviews, resolves conflicts, produces a unified review

**Key design decisions**:
- Agents run in parallel (speed) with a synthesizer at the end (coherence)
- Disagreement handling: if security reviewer says "block" and style reviewer says "approve," security always wins (priority ordering)
- Cost management: run cheap models for style/formatting, expensive models for security/logic
- Human escalation: if any reviewer flags "high-severity" issue, route to human reviewer before auto-approving

**Red flag**: Describes a single agent that "reviews everything." Does not consider specialization or disagreement handling.

**Follow-up**: "Two agents disagree on whether code is safe. How do you resolve?"
- Good: have a tie-breaking agent
- Great: depends on the domain. For security, default to "unsafe" (false positive is cheaper than false negative). For style, default to "approve" (nit-picks are not worth blocking). Have a clear priority ordering. If genuinely ambiguous, escalate to human with both agents' reasoning attached.

---

### Q2: "Your team wants to build a multi-agent system. Convince me it is better than a single agent."

**What this tests**: THIS IS A GOTCHA QUESTION. The interviewer wants you to push back.

**The right answer is often "it is not -- start simple."**

**Great answer**:
> "Multi-agent adds complexity: coordination overhead, increased cost, harder debugging, more failure modes. I would only use it when:
> (a) Tasks require genuinely different capabilities that cannot be combined in a single prompt (e.g., code review needs security expertise AND style expertise -- different system prompts, different tools)
> (b) Subtasks can run in parallel and the wall-clock time savings justify the coordination cost
> (c) The orchestration overhead is justified by measurable quality or speed gains
>
> For most tasks, a single agent with good tools is simpler, cheaper, and easier to debug. I would start with a single agent, measure where it fails, and only add agents for the specific failure modes that require specialized capabilities."

**Why this answer works**: It shows engineering judgment. Anyone can design a complex system. The hard part is knowing when complexity is not justified. Interviewers at senior/principal level are specifically testing whether you default to simplicity.

**Red flag**: Enthusiastically designs a 5-agent system without questioning whether it is needed. This is the most common mistake in multi-agent interviews.

---

### Q3: "Tell me about your experience building a multi-agent system" -- RESEARCH DEEP DIVE (45 min)

**What this tests**: This is the STAR project narrative. Practice this story until it is natural. Every detail matters.

**The narrative structure** (practice this exact flow):

1. **Problem** (2 min): "For a production service, manual log analysis for root cause analysis takes hours. Engineers spend much of their debugging time just reading and correlating logs across services."

2. **Why multi-agent** (3 min): "Logs regularly exceed single-model context limits, which is beyond any single model's context window. Different analysis stages need different expertise -- parsing raw logs requires different skills than correlating patterns across services or identifying root causes. A single agent could not handle both the scale and the variety."

3. **Architecture** (5 min): "We designed a three-tier system:
   - Task Planning agent decomposes the investigation into subtasks
   - Top-Level agents manage each investigation thread
   - RCA (Root Cause Analysis) agents do the deep dive on specific log segments
   - Map-reduce pattern for scalability: each agent processes a chunk, results are aggregated"

4. **Results** (2 min): "a major reduction in investigation time. a strong ROI when you factor in engineer hours saved. The system now handles investigations that previously took a senior engineer half a day in under 30 minutes."

5. **What you would do differently** (2 min): *[Prepare a genuine answer. Something like: "I would invest more in the evaluation harness earlier. I would also explore a more dynamic agent allocation -- our current system uses a fixed number of RCA agents, but some investigations need more than others."]*

6. **Extension** (if asked): "How would you add self-healing (auto-apply hotfixes)?"
   - This tests whether you can extend your own system. Good answer: "I would add a Fix Generation agent that proposes patches based on the RCA output, run them in a staging environment, and only auto-apply if (a) the fix passes all tests and (b) the issue is in a pre-approved category (e.g., config change, known error pattern). Anything novel goes to a human."

---
## Production Implementation Notes

*How multi-agent systems work at frontier labs -- the patterns they use and why.*

### Multi-Agent Patterns at Frontier Labs

**Anthropic -- Claude Code**:
- Uses a supervisor/worker pattern with tool specialization
- The supervisor agent decides what to do (plan), worker agents execute with different tool sets (file reading, code writing, terminal commands)
- Key insight: production systems often route planning to stronger (more expensive) models and execution to cheaper ones; the specific model assignments inside Claude Code are not public
- Handoff pattern: supervisor passes context + task description to worker, worker returns result, supervisor decides next step

**OpenAI -- Swarm Framework** (historical: an experimental/educational framework, superseded by the OpenAI Agents SDK in March 2025):
- Lightweight multi-agent framework focused on agent handoffs
- Key pattern: "routines" (system prompt + tools) that define an agent's capabilities
- Handoff is a function call: agent A calls `transfer_to_agent_b()` with context
- Designed for customer service workflows: triage agent -> billing agent -> technical agent
- Stateless by design -- all state is in the conversation history

**Google -- Gemini with Extensions** (dated branding; Google's current agent surface is Gemini with app integrations and the Agent Development Kit (ADK)):
- Uses a "function calling cascade" pattern
- Single model with many tools, rather than multiple specialized models
- The model itself decides which "extension" (tool set) to use
- Closer to single-agent-with-plugins than true multi-agent

### The Escalation Pattern (Most Important Production Pattern)

```
Level 0: Small model (Haiku/GPT-4o-mini) tries the task
  |-- Success? Done. Cost: $0.001
  |-- Failure? Escalate.
Level 1: Medium model (Sonnet/GPT-4o) tries with the error context
  |-- Success? Done. Cost: $0.01
  |-- Failure? Escalate.
Level 2: Large model (Opus/GPT-4) tries with full context
  |-- Success? Done. Cost: $0.10
  |-- Failure? Escalate to human.
Level 3: Human reviews with all agent attempts as context
  |-- Cost: $5-50 (human time)
```

In a well-tuned escalation stack, the large majority of tasks resolve at the cheapest levels (the exact split depends on your traffic mix), so the weighted average cost is much lower than always using the largest model.

### Cost Multiplier Reality

Multi-agent systems use **2-10x more tokens** than a single agent for the same task. This is because:
- Each agent needs its own system prompt (repeated tokens)
- Agents pass context to each other (duplicated information)
- Orchestration messages add overhead
- Failed agent attempts still cost money

**You must justify the ROI.** If multi-agent improves quality by 5% but costs 5x more, the single agent wins for most use cases. Multi-agent is justified when:
- Quality improvement is large (>20%)
- Tasks can be parallelized (wall-clock time drops despite more total compute)
- Different subtasks genuinely need different capabilities (not just different prompts)

### MCP: The Emerging Standard

Model Context Protocol (MCP) is the open standard for connecting AI models to external tools and data sources -- created by Anthropic, now under Linux Foundation ecosystem governance (Dec 2025). In multi-agent systems, MCP provides:
- Standardized tool descriptions that any agent can understand
- Consistent authentication and authorization across tools
- A registry pattern where agents discover available tools at runtime

In interviews, mentioning MCP shows awareness of the infrastructure layer that makes multi-agent systems practical. It is the "HTTP of agent tool use" -- not sexy, but essential for production.

---
## How This Gets Tested in Interviews

### The Multi-Agent Interview Is a Judgment Test

Unlike coding interviews that have a "right answer," multi-agent system design interviews test your **engineering judgment**. The interviewer is evaluating:

1. **Do you reach for multi-agent by default or by necessity?** Starting with "we need 5 agents" is a red flag. Starting with "let me first understand if a single agent with good tools would suffice" is a green flag.

2. **Can you decompose a problem into agent responsibilities?** If you use multi-agent, each agent should have a clear, non-overlapping responsibility. "Agent A handles X, Agent B handles Y" is good. "Agent A does everything, Agent B double-checks" is usually unjustified complexity.

3. **Do you think about failure and coordination?** What happens when Agent A fails? Does Agent B know? Can the system recover? Most candidates describe the happy path and never consider failures.

### A Multi-Agent System Narrative -- How to Structure It

This is your marquee project for multi-agent interviews. Practice these variations:

**2-minute version** (elevator pitch):
> Example: "A multi-agent system for automated root cause analysis. When logs exceed single-model context limits, a map-reduce architecture helps: a planning agent decomposes the investigation, specialized agents analyze log segments in parallel, and results are aggregated -- cutting diagnosis time substantially."

**5-minute version** (behavioral interview):
Add: the specific technical challenges (log parsing, context management, agent coordination), what you learned (evaluation is harder than building, start with metrics not vibes), and what you would do differently (dynamic agent allocation, better eval harness from day one).

**15-minute version** (deep dive):
Add: architecture diagrams (draw on whiteboard), specific failure modes and how you handled them, performance numbers (latency, cost, accuracy), comparison to baseline (manual investigation), and extension opportunities (self-healing, cross-service correlation).

### The "Single Agent vs Multi-Agent" Decision Framework

When pressed on this in an interview, use this framework:

```
Question: Does the task require >1 distinct capability set?
  NO  -> Single agent with good tools. Done.
  YES -> Can the capabilities fit in one system prompt?
    YES -> Single agent with role-switching. Done.
    NO  -> Multi-agent. But then:
      - Can subtasks run in parallel? -> Fan-out/fan-in pattern
      - Must subtasks run sequentially? -> Pipeline pattern
      - Do agents need to negotiate? -> Debate/consensus pattern
      - Is there a natural hierarchy? -> Supervisor/worker pattern
```

### Debugging Multi-Agent Systems

Interviewers love to ask: "How do you debug a multi-agent system when something goes wrong?"

**The answer**: Distributed tracing. Give every request a unique ID. Log every message between agents with the request ID, timestamps, and full content. Build a visualization that shows the message flow as a directed graph. When something fails, you can trace the exact sequence of messages that led to the failure.

This is essentially the same problem as debugging microservices -- and the same tools apply (Jaeger, Datadog, custom trace viewers). Mentioning this connection shows you understand that multi-agent systems are distributed systems with all the same challenges.

---
## 11. Flashcard Summary

| # | Question | Answer |
|---|---|---|
| 1 | Name 5 multi-agent architecture patterns | Supervisor/Worker, Debate/Consensus, Pipeline, Specialization (Team), Map-Reduce |
| 2 | When is multi-agent better than single agent? | When the task decomposes naturally, needs parallelism, requires diverse specialization, or exceeds a single context window |
| 3 | What is the Supervisor/Worker pattern? | One agent decomposes and delegates tasks to workers, then synthesizes their results. Centralized control. |
| 4 | What is the Debate/Consensus pattern? | Multiple agents independently analyze the same problem, then debate or vote to converge on the best answer. |
| 5 | What is the Pipeline pattern? | Agents in sequence, each transforms the output of the previous agent. Like Unix pipes. |
| 6 | What is the Map-Reduce pattern? | Split input into chunks, process in parallel, merge results. Key for large inputs. |
| 7 | Message passing vs shared memory? | Message passing: explicit, traceable, loosely coupled. Shared memory: implicit, flexible, tighter coupling. |
| 8 | What is MCP? | Model Context Protocol -- open standard for connecting AI agents to external tools/data. Developed by Anthropic. |
| 9 | What are MCP's three primitives? | Tools (actions), Resources (data), Prompts (templates). |
| 10 | How does a multi-agent RCA system use multi-agent patterns? | Supervisor/Worker for task decomposition + Map-Reduce for parallel log analysis, with 3 specialized agents. |
| 11 | Why 3 agents in a multi-agent RCA system, not 1? | Separation of concerns: planning, orchestration, and deep analysis are different skills needing different prompts/tools. |
| 12 | How to handle agent disagreement? | Majority vote, confidence weighting, judge agent, iterative debate, or ensemble (union of findings). |
| 13 | Main cost concern with multi-agent? | N system prompts x N conversation histories = multiplied token cost. Mitigate with mixed models and history limits. |
| 14 | When NOT to use multi-agent? | Simple tasks, latency-critical paths, when you can't observe/debug, when the task doesn't decompose. |
| 15 | What is agent handoff? | Transferring context (summary, findings, next steps) from one agent to another, enabling sequential collaboration without sharing full history. |

---
## 12. Paper Guides

### Paper 1: "Building Effective Agents" (Anthropic, 2024)

**Link**: https://www.anthropic.com/research/building-effective-agents

**Key Takeaways**:
- Start with the simplest agent architecture that works. Don't over-engineer.
- The "augmented LLM" (LLM + retrieval + tools) is the building block; multi-agent is the composition.
- **Workflow patterns** (deterministic orchestration) vs **agent patterns** (LLM decides the flow): most production systems should be workflows, not fully autonomous agents.
- Specific patterns discussed: prompt chaining, routing, parallelization, orchestrator-workers, evaluator-optimizer.
- When to use agents: open-ended problems where you can't predetermine the sequence of steps.

**Interview relevance**: This paper is from Anthropic. If you're interviewing there, know it well. The key insight is that **most production systems should use workflows (deterministic orchestration), not fully autonomous agents**.

**How multi-agent RCA aligns**: A multi-agent RCA system is a workflow (orchestrator-workers pattern), not a fully autonomous agent. The Task Planning Agent creates a deterministic plan, which is then executed. This is exactly what Anthropic recommends.

---

### Paper 2: "AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation" (Wu et al., 2023)

**Link**: https://arxiv.org/abs/2308.08155

**Key Takeaways**:
- Proposes a general framework for multi-agent conversations
- Key abstraction: `ConversableAgent` with configurable LLM, human input, and tool access
- Supports flexible conversation patterns: two-agent chat, group chat, hierarchical
- Demonstrates that multi-agent conversation can solve complex tasks better than single-agent
- Practical patterns: code generation + execution in separate agents, research + review loops

**What to focus on for interviews**:
- Section 2: The multi-agent conversation framework (architecture and abstractions)
- Section 3: Applications (shows the breadth of what multi-agent can do)
- Figure 1: The agent architecture diagram (be able to draw this on a whiteboard)

**Critical evaluation**: AutoGen is great for prototyping but adds complexity for production. For production systems (like a multi-agent RCA system), a simpler custom framework is often better because you control every aspect.

**Status update**: AutoGen merged with Semantic Kernel into the **Microsoft Agent Framework** (Oct 2025), which is now Microsoft's current agent framework.

---

### Bonus Reading

- **CrewAI documentation**: Popular framework for building multi-agent systems (practical reference)
- **LangGraph**: LangChain's graph-based multi-agent orchestration (production-focused)
- **"Communicative Agents for Software Development"** (Qian et al., 2023): ChatDev paper showing specialized agent teams for software engineering